In [10]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("--- TEST DE NOTRE SYSTEME ---")
# On essaie de lire le fichier situé dans le dossier data
df = pd.read_csv(r"F:\BD\hospital-readmission-project\data\diabetic_data.csv")
print("Bravo ! Le fichier est lu. Voici les 5 premières lignes :")
print(df.head(5))
print(f"Taille du fichier : {df.shape[0]} patients et {df.shape[1]} variables.\n")

--- TEST DE NOTRE SYSTEME ---
Bravo ! Le fichier est lu. Voici les 5 premières lignes :
   encounter_id  patient_nbr             race  gender      age weight  \
0       2278392      8222157        Caucasian  Female   [0-10)      ?   
1        149190     55629189        Caucasian  Female  [10-20)      ?   
2         64410     86047875  AfricanAmerican  Female  [20-30)      ?   
3        500364     82442376        Caucasian    Male  [30-40)      ?   
4         16680     42519267        Caucasian    Male  [40-50)      ?   

   admission_type_id  discharge_disposition_id  admission_source_id  \
0                  6                        25                    1   
1                  1                         1                    7   
2                  1                         1                    7   
3                  1                         1                    7   
4                  1                         1                    7   

   time_in_hospital  ... citoglipton insulin  

# PHASE 2 : L'AUDIT AGNOSTIQUE

In [11]:
print("--- 🔍 1. RECHERCHE DES CASES VIDES OFFICIELLES (NaN) ---")

# On compte les vraies cases vides pour chaque colonne
vides_officiels = df.isnull().sum()
vides_officiels = vides_officiels[vides_officiels > 0] # On ne garde que celles qui ont des trous

if vides_officiels.empty:
    print("Étonnant : Le système ne détecte AUCUNE case vide standard (NaN) dans tout le fichier.")
else:
    print(vides_officiels)
    
print("\n--- 🕵️‍♀️ 2. FOUILLE DES VALEURS (Top 3 par colonne) ---")
print("Analyse des 15 premières colonnes pour repérer les codes d'erreur de saisie...\n")

for colonne in df.columns[:15]:  # On se limite aux 15 premières colonnes pour l'audit
    print(f"👉 Analyse de la colonne : {colonne}")
    # On compte tout, même les vides (NaN)
    top_valeurs = df[colonne].value_counts(dropna=False).head(3)
    for valeur, compte in top_valeurs.items():
         print(f"   - {valeur} : {compte} fois")
    print("-" * 30)

--- 🔍 1. RECHERCHE DES CASES VIDES OFFICIELLES (NaN) ---
max_glu_serum    96420
A1Cresult        84748
dtype: int64

--- 🕵️‍♀️ 2. FOUILLE DES VALEURS (Top 3 par colonne) ---
Analyse des 15 premières colonnes pour repérer les codes d'erreur de saisie...

👉 Analyse de la colonne : encounter_id
   - 2278392 : 1 fois
   - 190792044 : 1 fois
   - 190790070 : 1 fois
------------------------------
👉 Analyse de la colonne : patient_nbr
   - 88785891 : 40 fois
   - 43140906 : 28 fois
   - 1660293 : 23 fois
------------------------------
👉 Analyse de la colonne : race
   - Caucasian : 76099 fois
   - AfricanAmerican : 19210 fois
   - ? : 2273 fois
------------------------------
👉 Analyse de la colonne : gender
   - Female : 54708 fois
   - Male : 47055 fois
   - Unknown/Invalid : 3 fois
------------------------------
👉 Analyse de la colonne : age
   - [70-80) : 26068 fois
   - [60-70) : 22483 fois
   - [50-60) : 17256 fois
------------------------------
👉 Analyse de la colonne : weight
   - ? : 

PHASE 2bis : PURGE DES COLONNES TROP VIDES

In [12]:
seuil_limite = 0.60  # On supprime si > 60% de '?'

colonnes_a_supprimer = []

for col in df.columns:
    taux_manquant = (df[col] == '?').mean()
    if taux_manquant > seuil_limite:
        colonnes_a_supprimer.append(col)

print(f"Colonnes supprimées car trop vides (>60%) : {colonnes_a_supprimer}")
df = df.drop(columns=colonnes_a_supprimer)

import numpy as np # On importe numpy pour gérer les vrais 'vides'

Colonnes supprimées car trop vides (>60%) : ['weight']


PHASE 3 : HARMONISATION

In [13]:
print("\n--- 🧹 NETTOYAGE DES DONNÉES ---")

# 1. On remplace tous les '?' par des vrais vides (NaN) que Python comprend
df = df.replace('?', np.nan)

# 2. On traite les cas 'Unknown/Invalid' dans gender pour les mettre à NaN
# (car 3 cas sur 100 000, c'est du bruit statistique inutile)
df['gender'] = df['gender'].replace('Unknown/Invalid', np.nan)

# 3. Maintenant, on peut compter les vrais vides NaN proprement
total_vides = df.isnull().sum().sum()
print(f"Nettoyage terminé. Total de valeurs manquantes (NaN) dans le fichier : {total_vides}")

# Petit contrôle sur le genre pour être sûre que les 'Unknown/Invalid' ont bien disparu
# 3. On affiche la répartition de la colonne gender pour voir le 'Unknown'
print("\nRépartition de la colonne 'gender' :")
print(df['gender'].value_counts(dropna=False))


--- 🧹 NETTOYAGE DES DONNÉES ---
Nettoyage terminé. Total de valeurs manquantes (NaN) dans le fichier : 275451

Répartition de la colonne 'gender' :
gender
Female    54708
Male      47055
NaN           3
Name: count, dtype: int64


# PHASE 4 : PRÉPARATION DE LA CIBLE

In [14]:
print("\n--- 🎯 PRÉPARATION DE LA CIBLE (readmitted) ---")

# On crée une nouvelle colonne binaire : 1 si réadmis, 0 sinon
df['is_readmitted'] = df['readmitted'].apply(lambda x: 0 if x == 'NO' else 1)

# Calcul de la répartition en pourcentages (normalize=True)
stats_readmission = df['is_readmitted'].value_counts(normalize=True) * 100

# Vérification
print("Répartition de la cible (0 = Pas réadmis, 1 = Réadmis) :")
print(df['is_readmitted'].value_counts())
print(f" - Pas réadmis (0) : {stats_readmission[0]:.1f} %")
print(f" - Réadmis (1)      : {stats_readmission[1]:.1f} %")


--- 🎯 PRÉPARATION DE LA CIBLE (readmitted) ---
Répartition de la cible (0 = Pas réadmis, 1 = Réadmis) :
is_readmitted
0    54864
1    46902
Name: count, dtype: int64
 - Pas réadmis (0) : 53.9 %
 - Réadmis (1)      : 46.1 %


PHASE 5 : ANALYSE DES CORRÉLATIONS

In [15]:
print("\n--- 📈 ANALYSE DES FACTEURS DE RÉADMISSION ---")

# On sélectionne uniquement les colonnes numériques pour le calcul
# (C'est mathématiquement plus simple pour débuter)
numeric_df = df.select_dtypes(include=['number'])

# On calcule la corrélation avec notre cible
correlations = numeric_df.corr()['is_readmitted'].sort_values(ascending=False)

print("Top 10 des facteurs qui influencent le plus la réadmission :")
print(correlations.head(11)) # On affiche 11 pour voir la cible elle-même en premier


--- 📈 ANALYSE DES FACTEURS DE RÉADMISSION ---
Top 10 des facteurs qui influencent le plus la réadmission :
is_readmitted          1.000000
number_inpatient       0.217194
number_diagnoses       0.112564
number_emergency       0.103011
number_outpatient      0.082142
patient_nbr            0.074093
time_in_hospital       0.051289
num_medications        0.046772
admission_source_id    0.039986
num_lab_procedures     0.039253
admission_type_id     -0.004923
Name: is_readmitted, dtype: float64


PHASE 6.1 : REDUCTION DE DIMENSION
PHASE 6.1 : ARBITRAGE POUR NUM_MEDICATIONS

In [ ]:


print("\n--- ⚖️ TEST D'AFFINITÉ POUR NUM_MEDICATIONS ---")

# On teste la corrélation de num_medications avec les "chefs de file" des deux dimensions
corr_avec_terrain = df['num_medications'].corr(df['number_diagnoses'])
corr_avec_crise = df['num_medications'].corr(df['time_in_hospital'])

print(f"Affinité avec le Terrain (number_diagnoses) : {corr_avec_terrain:.3f}")
print(f"Affinité avec la Crise (time_in_hospital)   : {corr_avec_crise:.3f}")

if corr_avec_crise > corr_avec_terrain:
    print("👉 Conclusion data : num_medications reflète davantage la Sévérité de l'épisode (Dim 3).")
else:
    print("👉 Conclusion data : num_medications reflète davantage le Fardeau Pathologique (Dim 1).")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# On crée une figure avec deux sous-graphiques côte à côte
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Graphique 1 : Médicaments vs Terrain
sns.regplot(data=df, x='number_diagnoses', y='num_medications', ax=axes[0], 
            scatter_kws={'alpha':0.1}, line_kws={'color':'red'})
axes[0].set_title(f"Affinité Terrain (r={corr_avec_terrain:.3f})")

# Graphique 2 : Médicaments vs Crise
sns.regplot(data=df, x='time_in_hospital', y='num_medications', ax=axes[1], 
            scatter_kws={'alpha':0.1}, line_kws={'color':'green'})
axes[1].set_title(f"Affinité Crise (r={corr_avec_crise:.3f})")

plt.suptitle("Arbitrage Clinique : Où placer 'num_medications' ?", fontsize=16)
plt.show()

PHASE 6 : INGÉNIERIE DES 3 DIMENSIONS CLINIQUES

In [19]:
from sklearn.preprocessing import MinMaxScaler

print("\n--- 🧬 CRÉATION DES 3 DIMENSIONS CLINIQUES ---")

scaler = MinMaxScaler()

# Dim 1 : Terrain pathologique 
# (On duplique la colonne pour garder un nommage cohérent)
df['dim_terrain'] = df['number_diagnoses']

# Dim 2 : Instabilité chronique (Historique)
cols_instab = ['number_inpatient', 'number_emergency', 'number_outpatient']
poids_instab = [0.217, 0.103, 0.082]   # les r de corrélation

normed_instab = pd.DataFrame(
    scaler.fit_transform(df[cols_instab]),
    columns=cols_instab,
    index=df.index
)
df['dim_instabilite'] = sum(normed_instab[c] * w for c, w in zip(cols_instab, poids_instab))

# Dim 3 : Sévérité de l'épisode actuel
# On intègre num_medications ici suite au test d'affinité statistique
cols_sev = ['time_in_hospital', 'num_lab_procedures', 'num_medications']
poids_sev = [0.051, 0.039, 0.047]

normed_sev = pd.DataFrame(
    scaler.fit_transform(df[cols_sev]),
    columns=cols_sev,
    index=df.index
)
df['dim_severite'] = sum(normed_sev[c] * w for c, w in zip(cols_sev, poids_sev))


# On isole nos variables finales (X) et notre cible (y) pour préparer l'IA
features_finales = ['dim_terrain', 'dim_instabilite', 'dim_severite']
X = df[features_finales]
y = df['is_readmitted']

# Petite vérification statistique finale
print("Puissance prédictive de nos 3 axes cliniques :")
correlations_finales = df[features_finales + ['is_readmitted']].corr()['is_readmitted'] * 100
for col, score in correlations_finales.items():
    if col != 'is_readmitted':
        print(f"👉 {col:20} : {score:+.1f}%")

from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report



--- 🧬 CRÉATION DES 3 DIMENSIONS CLINIQUES ---
Puissance prédictive de nos 3 axes cliniques :
👉 dim_terrain          : +11.3%
👉 dim_instabilite      : +22.7%
👉 dim_severite         : +6.0%


GRAPHIQUE 1 : LA MATRICE DE CORRÉLATION

In [ ]:
plt.figure(figsize=(10, 8))
cols_analyse = ['number_inpatient', 'number_emergency', 'number_outpatient', 
                'time_in_hospital', 'num_lab_procedures', 'num_medications', 
                'number_diagnoses', 'is_readmitted']
corr_matrix = df[cols_analyse].corr()

sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title("Matrice de Corrélation : Justification Clinique des Dimensions", fontsize=15, pad=20)
plt.show()

PHASE 7 : MACHINE LEARNING (Forêt Aléatoire)

In [ ]:
print("\n--- 🤖 ENTRAÎNEMENT DE L'INTELLIGENCE ARTIFICIELLE ---")

# 1. Préparation de la matrice d'apprentissage (X) et de la cible (y)
features_finales = ['dim_terrain', 'dim_instabilite', 'dim_severite']
X = df[features_finales]
y = df['is_readmitted']

# 2. Le Découpage Stratégique (80% pour apprendre, 20% pour l'examen final)
# stratify=y garantit que les 20% de test auront la même proportion de diabétiques réadmis que la vraie vie.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    random_state=42, 
    stratify=y
)

print(f"📚 Apprentissage + Validation croisée sur : {X_train.shape[0]} patients")
print(f"🎯 Examen final sur  : {X_test.shape[0]} patients cachés\n")

In [ ]:
# --- GRAPHIQUE 2 : BOXPLOT DE LA SÉVÉRITÉ (Nettoyage & Outliers) ---
# Ce graphique montre la distribution de ta Dim 3 et comment elle impacte la réadmission
plt.figure(figsize=(10, 6))
sns.boxplot(x='is_readmitted', y='dim_severite', data=df, palette='Set2')
plt.title("Distribution de la Sévérité de l'Épisode (Dim 3) vs Réadmission", fontsize=14)
plt.xlabel("Réadmis (0 = Non, 1 = Oui)")
plt.ylabel("Score de Sévérité Cumulé")
plt.show()

In [ ]:
print("\n--- 🤖 ENTRAÎNEMENT DE L'INTELLIGENCE ARTIFICIELLE ---")

# 1. Préparation de la matrice d'apprentissage (X) et de la cible (y)
features_finales = ['dim_terrain', 'dim_instabilite', 'dim_severite']
X = df[features_finales]
y = df['is_readmitted']

# 2. Le Découpage Stratégique (80% pour apprendre, 20% pour l'examen final)
# stratify=y garantit que les 20% de test auront la même proportion de diabétiques réadmis que la vraie vie.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    random_state=42, 
    stratify=y
)

print(f"📚 Apprentissage + Validation croisée sur : {X_train.shape[0]} patients")
print(f"🎯 Examen final sur  : {X_test.shape[0]} patients cachés\n")

# 3. Création du Cerveau (L'Algorithme)
# On limite la profondeur à 10 (max_depth) pour éviter qu'il n'apprenne les dossiers par cœur
print("⏳ Croissance des 100 arbres de décision en cours...")
model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)

# 4. LA VALIDATION CROISÉE (K-Fold = 5)
print("⏳ Lancement de la Validation Croisée (5 Folds)...")
# cv=5 : On coupe le X_train en 5, on entraîne 5 fois.
# scoring='roc_auc' : On demande l'évaluation médicale standard.
scores_cv = cross_val_score(model, X_train, y_train, cv=5, scoring='roc_auc')

print(f"📊 Scores des 5 examens blancs : {scores_cv}")
print(f"🏆 Score ROC-AUC moyen sur les 5 folds / Score moyen de Validation : {scores_cv.mean():.3f} (Stabilité: ±{scores_cv.std():.3f})\n")

# 5. L'EXAMEN FINAL (Sur le jeu de test de 20%) / L'Apprentissage final du modèle sur tout le Train
# On doit d'abord entraîner le modèle officiellement sur l'intégralité du Train
print("⏳ Entraînement final du modèle sur tout le jeu d'apprentissage...")
model.fit(X_train, y_train)

# 5. L'Examen Final (Sur les données jamais vues)
# On demande à l'IA de donner un pourcentage de risque pour chaque patient du Test
y_pred_proba = model.predict_proba(X_test)[:, 1] 

# Le Score ROC-AUC (La note officielle en biostatistiques)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print(f"✅ EXAMEN TERMINÉ !")
print(f"🏆 Score ROC-AUC : {roc_auc:.3f}")
print("(0.50 = Pile ou face | 1.00 = Perfection)")        

GRAPHIQUE 3 : VIOLIN PLOT DE L'INSTABILITÉ (Le signal majeur)

 PHASE 8 : EXPLICABILITÉ DU MODÈLE (Feature Importance)

In [ ]:
print("\n--- 🔍 CE QUE L'IA A COMPRIS (Poids des Dimensions) ---")

# On extrait les pourcentages d'importance calculés par la Forêt Aléatoire
importances = model.feature_importances_ * 100
colonnes = X_train.columns

# On crée un beau tableau trié du plus important au moins important
df_importances = pd.DataFrame({
    'Dimension Clinique': colonnes,
    'Importance dans la décision (%)': importances
}).sort_values(by='Importance dans la décision (%)', ascending=False)

# Affichage propre
print(df_importances.to_string(index=False))
print("\n(L'addition de ces poids fait toujours 100%)")

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve

In [ ]:
# Pour montrer pourquoi l'instabilité (65% d'importance) est le moteur du modèle
plt.figure(figsize=(10, 6))
sns.violinplot(x='is_readmitted', y='dim_instabilite', data=df, inner="quartile", palette="Pastel1")
plt.title("Impact de l'Instabilité Chronique (Dim 2) sur le Risque", fontsize=14)
plt.xlabel("Réadmis (0 = Non, 1 = Oui)")
plt.ylabel("Nombre de contacts antérieurs")
plt.show()

 PHASE 9 : DATA STORYTELLING (Visualisation)

In [ ]:
print("\n--- 📊 GÉNÉRATION DES GRAPHIQUES ---")

# Style global professionnel
sns.set_theme(style="whitegrid")
plt.figure(figsize=(15, 6))

# GRAPHIQUE 1 : L'Importance des Variables (Ce que l'IA regarde)
plt.subplot(1, 2, 1)
# On utilise df_importances créé à la phase 8
sns.barplot(
    x='Importance dans la décision (%)', 
    y='Dimension Clinique', 
    data=df_importances, 
    palette='viridis'
)
plt.title("Qu'est-ce qui déclenche une réadmission ?", fontsize=14, fontweight='bold')
plt.xlabel("Poids dans la décision de l'IA (%)")
plt.ylabel("")

# GRAPHIQUE 2 : La Courbe ROC (La performance médicale)
plt.subplot(1, 2, 2)
# Calcul des points de la courbe ROC
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)

plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'Modèle IA (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Hasard complet (AUC = 0.5)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Fausses alertes (Taux de Faux Positifs)')
plt.ylabel('Malades détectés (Sensibilité)')
plt.title("Performance Clinique (Courbe ROC)", fontsize=14, fontweight='bold')
plt.legend(loc="lower right")

# Affichage de la fenêtre avec les graphiques
plt.tight_layout()
plt.show()


import matplotlib.pyplot as plt
import seaborn as sns

# ... (ton code de calcul et de préparation du df_importances) ...

# 1. GRAPHIQUE : Importance des variables
plt.figure(figsize=(10, 6))
sns.barplot(
    x='Importance dans la décision (%)', 
    y='Dimension Clinique', 
    data=df_importances, 
    palette='viridis'
)
plt.title("Importance des dimensions cliniques")
plt.tight_layout()

# C'est cette ligne qui fait apparaître la fenêtre !
plt.show() 

# 2. GRAPHIQUE : Courbe ROC
plt.figure(figsize=(8, 8))
# ... (ton code de courbe ROC) ...
plt.plot([0, 1], [0, 1], 'k--')
plt.title("Courbe ROC")

# Très important : le deuxième show() fera apparaître la seconde fenêtre
plt.show()